# Comparación de modelos y aplicación a los proveedores sin órdenes (HU 6.3)
**Autor:** Sebas · **Entradas (en `data/processed/`):** `oof_predicciones.csv`, `resultados_validacion.csv`, `modelado_train.csv`, `modelado_pred.csv`, `modelos_epic6.joblib`

Objetivo: comparar los dos modelos de la HU 6.2 contra las líneas base, elegir el mejor, aplicarlo a los 18 proveedores sin órdenes y documentar la importancia de las variables y las limitaciones.

**Salidas:** `tabla_comparativa_modelos.csv`, `matriz_confusion_<modelo>.csv`, `prediccion_18_proveedores.csv`, `importancia_variables.csv` y `justificacion_mejor_modelo.md` (en `data/processed/`), más las figuras en `reports/figures/`.

**Qué hace la celda siguiente:** importa las librerías, define las rutas, la semilla y el orden de las clases, y carga los archivos que dejó Miguel. Luego verifica que `oof_predicciones.csv` esté completo.

**Qué significa:** cada modelo debe tener 9 evaluaciones (3 repeticiones × 3 particiones) y cada proveedor debe aparecer 3 veces por modelo (una por repetición). Si alguna verificación falla, hay que avisarle a Miguel antes de seguir.

In [2]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)

ROOT = Path('..') if Path('../data').exists() else Path('.')
PROC = ROOT / 'data/processed'
FIG = ROOT / 'reports/figures'; FIG.mkdir(parents=True, exist_ok=True)
SEED = 42
CLASES = ['Grandes recurrentes', 'Pequeños recurrentes', 'Ocasionales']
MODELOS = ['mayoritaria', 'estratificado', 'regresion_logistica', 'random_forest']
NOMBRE = {'mayoritaria': 'Línea base: clase mayoritaria', 'estratificado': 'Línea base: azar estratificado',
          'regresion_logistica': 'Regresión logística', 'random_forest': 'Random Forest'}

oof = pd.read_csv(PROC / 'oof_predicciones.csv')
res_07 = pd.read_csv(PROC / 'resultados_validacion.csv', index_col=0)
train = pd.read_csv(PROC / 'modelado_train.csv')
pred = pd.read_csv(PROC / 'modelado_pred.csv')
art = joblib.load(PROC / 'modelos_epic6.joblib')

# --- Verificaciones de integridad ---
assert set(oof['modelo']) == set(MODELOS), f"Modelos inesperados: {set(oof['modelo'])}"
n_eval = oof.groupby('modelo').apply(lambda g: g[['repeticion', 'pliegue']].drop_duplicates().shape[0])
veces = oof.groupby(['modelo', 'VendorID']).size().groupby('modelo').agg(['min', 'max'])
print('Evaluaciones por modelo (esperado 9):'); print(n_eval.to_dict())
print('Veces que aparece cada proveedor por modelo (esperado min=max=3):'); print(veces)
print('Proveedores de entrenamiento:', train.shape[0], '| a predecir:', pred.shape[0])
print('Distribución del segmento real:'); print(train['Segmento'].value_counts())
print('Variables del modelo:', art['features'], '| excluidos:', art['excluidos'])

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\processed\\oof_predicciones.csv'

## TASK 6.3.1 – Evaluar los modelos (#66)

### Precisión y F1 macro por partición
**Qué hace la celda siguiente:** calcula la precisión y el F1 macro de cada combinación de modelo, repetición y partición, y luego promedia (media y desviación estándar, la misma `std` poblacional que usó Miguel). Compara el resultado con `resultados_validacion.csv`.

**Qué significa:** las dos tablas deben coincidir. Si no coinciden, hay una diferencia entre este notebook y el 07 que hay que aclarar con Miguel.

In [ ]:
def metricas(g):
    return pd.Series({'accuracy': accuracy_score(g['y_real'], g['y_pred']),
                      'f1_macro': f1_score(g['y_real'], g['y_pred'], average='macro')})

por_pliegue = (oof.groupby(['modelo', 'repeticion', 'pliegue']).apply(metricas).reset_index())
std0 = lambda s: s.std(ddof=0)   # igual que np.std en el notebook 07
resumen = (por_pliegue.groupby('modelo')
           .agg(accuracy_media=('accuracy', 'mean'), accuracy_std=('accuracy', std0),
                f1_macro_media=('f1_macro', 'mean'), f1_macro_std=('f1_macro', std0))
           .loc[MODELOS])
display(resumen)

cols = list(resumen.columns)
coincide = np.allclose(resumen[cols].values, res_07.loc[MODELOS, cols].values, atol=1e-6)
print('¿Coincide con resultados_validacion.csv?', 'SÍ' if coincide else 'NO -> avisar a Miguel')

### Matriz de confusión de cada modelo
**Qué hace la celda siguiente:** calcula la matriz de confusión de cada modelo con todas sus filas de `oof_predicciones.csv`, normalizada por fila (`normalize='true'`) y con las clases en el orden `Grandes recurrentes`, `Pequeños recurrentes`, `Ocasionales`. Dibuja las cuatro y guarda cada una en CSV.

**Qué significa:** cada fila es el segmento real y cada columna el segmento predicho; el número es la proporción de proveedores de ese segmento que cayó en cada columna. La diagonal es el acierto por clase (*recall*). Cada proveedor aparece 3 veces por modelo (una por repetición), por eso las Ocasionales suman 21 filas, no 7.

In [ ]:
cms = {}
for m in MODELOS:
    g = oof[oof['modelo'] == m]
    cms[m] = pd.DataFrame(confusion_matrix(g['y_real'], g['y_pred'], labels=CLASES, normalize='true'),
                          index=[f'real: {c}' for c in CLASES], columns=[f'pred: {c}' for c in CLASES])
    cms[m].to_csv(PROC / f'matriz_confusion_{m}.csv')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, m in zip(axes.ravel(), MODELOS):
    cm = cms[m].values
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASES, rotation=20, ha='right')
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > 0.5 else 'black')
    ax.set_title(NOMBRE[m]); ax.set_xlabel('Segmento predicho'); ax.set_ylabel('Segmento real')
plt.tight_layout(); plt.savefig(FIG / 'matrices_confusion_epic6.png', dpi=150); plt.show()

recall = pd.DataFrame({NOMBRE[m]: np.diag(cms[m].values) for m in MODELOS}, index=CLASES)
print('Acierto por clase (diagonal de la matriz normalizada):'); display(recall)
print('Filas reales por clase en cada modelo:', oof[oof['modelo'] == 'random_forest']['y_real'].value_counts().to_dict())

### Comparación contra las líneas base
**Qué hace la celda siguiente:** calcula, para cada modelo, la diferencia de precisión y de F1 macro frente a las dos líneas base, y cuenta en cuántas de las 9 evaluaciones el modelo supera a la clase mayoritaria (precisión) y al azar estratificado (F1 macro).

**Qué significa:** un modelo solo aporta información si supera a la clase mayoritaria en precisión **y** al azar en F1 macro. Una diferencia pequeña frente a la desviación estándar no es evidencia de que el modelo aprenda.

In [ ]:
R = resumen
filas = {}
for m in ['regresion_logistica', 'random_forest']:
    filas[NOMBRE[m]] = {
        'Δ precisión vs mayoritaria': R.loc[m, 'accuracy_media'] - R.loc['mayoritaria', 'accuracy_media'],
        'Δ precisión vs estratificado': R.loc[m, 'accuracy_media'] - R.loc['estratificado', 'accuracy_media'],
        'Δ F1 macro vs mayoritaria': R.loc[m, 'f1_macro_media'] - R.loc['mayoritaria', 'f1_macro_media'],
        'Δ F1 macro vs estratificado': R.loc[m, 'f1_macro_media'] - R.loc['estratificado', 'f1_macro_media']}
display(pd.DataFrame(filas).T)

pp = por_pliegue.pivot_table(index=['repeticion', 'pliegue'], columns='modelo', values=['accuracy', 'f1_macro'])
for m in ['regresion_logistica', 'random_forest']:
    gana_acc = (pp['accuracy'][m] > pp['accuracy']['mayoritaria']).sum()
    gana_f1 = (pp['f1_macro'][m] > pp['f1_macro']['estratificado']).sum()
    print(f"{NOMBRE[m]}: supera a la mayoritaria en precisión en {gana_acc}/9 evaluaciones; "
          f"supera al azar estratificado en F1 macro en {gana_f1}/9.")

## TASK 6.3.2 – Tabla comparativa y mejor modelo (#67)

**Qué hace la celda siguiente:** arma la tabla comparativa de los cuatro modelos con precisión y F1 macro (media ± desviación) y la guarda en `tabla_comparativa_modelos.csv`.

**Qué significa:** esta es la tabla que pide el criterio de aceptación. Se reporta el F1 macro junto con la precisión porque la clase mayoritaria obtiene una precisión alta sin aprender nada.

In [ ]:
tabla_comp = pd.DataFrame({
    'Modelo': [NOMBRE[m] for m in MODELOS],
    'Precisión (media ± desv.)': [f"{R.loc[m,'accuracy_media']:.3f} ± {R.loc[m,'accuracy_std']:.3f}" for m in MODELOS],
    'F1 macro (media ± desv.)': [f"{R.loc[m,'f1_macro_media']:.3f} ± {R.loc[m,'f1_macro_std']:.3f}" for m in MODELOS],
    'Acierto en Ocasionales': [recall.loc['Ocasionales', NOMBRE[m]] for m in MODELOS]})
tabla_comp.to_csv(PROC / 'tabla_comparativa_modelos.csv', index=False)
display(tabla_comp)

**Qué hace la celda siguiente:** elige el mejor de los dos modelos por precisión (desempate por F1 macro y luego por menor desviación), y revisa tres cosas: si la diferencia entre los modelos es menor que la desviación estándar, si el elegido supera a las líneas base y si lo supera con claridad (diferencia mayor que su propia desviación). Escribe la justificación y la guarda en `justificacion_mejor_modelo.md`.

**Qué significa:** la selección es mecánica, pero la lectura importa: si no hay ganador claro o ningún modelo supera las líneas base, la justificación lo dice de forma explícita (es un resultado válido y una limitación, según el criterio de aceptación).

In [ ]:
CAND = ['regresion_logistica', 'random_forest']
orden = R.loc[CAND].sort_values(['accuracy_media', 'f1_macro_media', 'accuracy_std'], ascending=[False, False, True])
mejor, otro = orden.index[0], orden.index[1]

dif_acc = R.loc[mejor, 'accuracy_media'] - R.loc[otro, 'accuracy_media']
dif_f1 = R.loc[mejor, 'f1_macro_media'] - R.loc[otro, 'f1_macro_media']
std_ref = max(R.loc[CAND, 'accuracy_std'])
sin_ganador = dif_acc < std_ref

def evalua_base(m):
    d_acc = R.loc[m, 'accuracy_media'] - R.loc['mayoritaria', 'accuracy_media']
    d_f1 = R.loc[m, 'f1_macro_media'] - R.loc['estratificado', 'f1_macro_media']
    supera = (d_acc > 0) and (d_f1 > 0)
    claro = supera and (d_acc > R.loc[m, 'accuracy_std']) and (d_f1 > R.loc[m, 'f1_macro_std'])
    return d_acc, d_f1, supera, claro

est = {m: evalua_base(m) for m in CAND}
supera_base, supera_claro = est[mejor][2], est[mejor][3]
ninguno_supera = not any(est[m][2] for m in CAND)

L = []
L.append(f"**Modelo elegido: {NOMBRE[mejor]}**, por mayor precisión media ({R.loc[mejor,'accuracy_media']:.3f} "
         f"frente a {R.loc[otro,'accuracy_media']:.3f} de {NOMBRE[otro]}); F1 macro {R.loc[mejor,'f1_macro_media']:.3f} "
         f"frente a {R.loc[otro,'f1_macro_media']:.3f}; desviación de la precisión {R.loc[mejor,'accuracy_std']:.3f} "
         f"frente a {R.loc[otro,'accuracy_std']:.3f}.")
if sin_ganador:
    L.append(f"**No hay un ganador claro:** la diferencia de precisión entre los dos modelos ({dif_acc:.3f}) es menor que la "
             f"desviación estándar ({std_ref:.3f}). La elección es nominal y podría cambiar con otra semilla.")
else:
    L.append(f"La diferencia de precisión entre los modelos ({dif_acc:.3f}) supera la desviación estándar ({std_ref:.3f}), "
             f"aunque con 86 proveedores y 3 repeticiones sigue siendo una diferencia modesta.")
for m in CAND:
    d_acc, d_f1, ok, claro = est[m]
    L.append(f"{NOMBRE[m]}: precisión {d_acc:+.3f} vs la clase mayoritaria ({R.loc['mayoritaria','accuracy_media']:.3f}); "
             f"F1 macro {d_f1:+.3f} vs el azar estratificado ({R.loc['estratificado','f1_macro_media']:.3f}) -> "
             + ('supera ambas líneas base' + (' con claridad.' if claro else ', pero sin un margen claro frente a la desviación.')
                if ok else 'NO supera ambas líneas base.'))
if ninguno_supera:
    L.append("**Ningún modelo supera a la clase mayoritaria en precisión y al azar estratificado en F1 macro.** Es un resultado válido: "
             "el segmento se define por monto y número de órdenes, y las variables del proveedor (calificación, preferido, activo, "
             "provincia, URL) casi no lo explican. Las asignaciones a los 18 proveedores no tienen respaldo predictivo.")
elif not supera_claro:
    L.append("El modelo elegido no supera con claridad las líneas base: debe presentarse como limitación y el equipo decide cómo mostrar las asignaciones.")
JUSTIFICACION = '\n\n'.join(L)
(PROC / 'justificacion_mejor_modelo.md').write_text(JUSTIFICACION, encoding='utf-8')
print(JUSTIFICACION)

## TASK 6.3.3 – Aplicar el mejor modelo a los 18 proveedores (#68)

**Qué hace la celda siguiente:** carga del `.joblib` el modelo elegido en 6.3.2, calcula la probabilidad de cada segmento para los 18 proveedores, toma la mayor como `probabilidad` del segmento estimado y arma la tabla con las marcas de baja confianza. Guarda `prediccion_18_proveedores.csv`.

**Qué significa:** hay dos marcas. `baja_confianza_umbral` es verdadera si la probabilidad es menor que el umbral propuesto (0,60; el equipo debe confirmarlo). `baja_confianza` es la marca final: también es verdadera para todos los proveedores si el modelo no supera las líneas base, porque entonces ni siquiera una probabilidad alta es evidencia de acierto.

In [ ]:
UMBRAL = 0.60   # propuesta inicial; el equipo debe confirmarlo
clave = {'regresion_logistica': 'lr', 'random_forest': 'rf'}[mejor]
modelo = art[clave]
X = pred[art['features']]
proba = pd.DataFrame(modelo.predict_proba(X), columns=[f'prob_{c}' for c in modelo.classes_])

out = pred[['VendorID', 'NombreProveedor']].copy()
out['segmento_estimado'] = modelo.predict(X)
out['probabilidad'] = proba.max(axis=1).round(3).values
out['baja_confianza_umbral'] = out['probabilidad'] < UMBRAL
out['baja_confianza'] = out['baja_confianza_umbral'] | (not supera_base)
out['motivo_baja_confianza'] = np.select(
    [~out['baja_confianza'],
     (not supera_base) & out['baja_confianza_umbral'],
     (not supera_base)],
    ['', f'Probabilidad < {UMBRAL} y el modelo no supera la línea base',
     'El modelo no supera la línea base'],
    default=f'Probabilidad < {UMBRAL}')
out = pd.concat([out, proba.round(3)], axis=1)

assert len(out) == 18 and out['VendorID'].is_unique and out[['segmento_estimado', 'probabilidad']].notna().all().all()
out.to_csv(PROC / 'prediccion_18_proveedores.csv', index=False)
print(f'Modelo aplicado: {NOMBRE[mejor]} | umbral: {UMBRAL} | modelo supera líneas base: {supera_base}')
print('Baja confianza (final):', int(out['baja_confianza'].sum()), 'de', len(out),
      '| solo por umbral:', int(out['baja_confianza_umbral'].sum()))
print(out['segmento_estimado'].value_counts())
display(out)

**Qué hace la celda siguiente:** muestra cómo son los 18 proveedores en las variables del modelo.

**Qué significa:** los 18 no tienen ninguna orden, así que el segmento estimado es una extrapolación a partir de variables que casi no varían. Si casi todos comparten los mismos valores, el modelo les asigna casi lo mismo.

In [ ]:
for c in ['CreditRating', 'PreferredVendorStatus', 'ActiveFlag', 'ProvinciaGrupo', 'TieneURL']:
    if c in pred.columns:
        print(pred[c].value_counts(dropna=False).to_dict(), '<-', c)

## TASK 6.3.4 – Importancia de las variables y limitaciones (#69)

**Qué hace la celda siguiente:** calcula tres medidas de importancia para el modelo elegido y para el otro: la **importancia por permutación** (cuánto baja el F1 macro al desordenar una variable; es la más fiable), la importancia interna del Random Forest y los coeficientes de la regresión logística. Guarda `importancia_variables.csv` y la figura.

**Qué significa:** la importancia por permutación se calcula aquí sobre los 86 datos con que se entrenó el modelo, así que tiende a ser optimista (el modelo ya vio esos datos). Sirve para comparar variables entre sí, no para afirmar que el modelo generaliza. No se espera una variable dominante; si ninguna destaca, eso es consistente con que el modelo no supera las líneas base.

In [ ]:
FEATS = art['features']
Xtr, ytr = train[FEATS], train['Segmento']

imp = {}
for m, k in [('regresion_logistica', 'lr'), ('random_forest', 'rf')]:
    pi = permutation_importance(art[k], Xtr, ytr, scoring='f1_macro', n_repeats=30, random_state=SEED)
    imp[m] = pd.DataFrame({'variable': FEATS, 'importancia_media': pi.importances_mean,
                           'importancia_std': pi.importances_std}).sort_values('importancia_media', ascending=False)
    imp[m].insert(0, 'modelo', m)
    print(NOMBRE[m], '- importancia por permutación (caída del F1 macro):'); display(imp[m].drop(columns='modelo'))
pd.concat(imp.values()).to_csv(PROC / 'importancia_variables.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, m in zip(axes, imp):
    d = imp[m].iloc[::-1]
    ax.barh(d['variable'], d['importancia_media'], xerr=d['importancia_std'], color='#4C78A8')
    ax.axvline(0, color='gray', lw=0.8); ax.set_title(NOMBRE[m]); ax.set_xlabel('Caída del F1 macro al permutar')
plt.tight_layout(); plt.savefig(FIG / 'importancia_permutacion_epic6.png', dpi=150); plt.show()

rf_pipe, lr_pipe = art['rf'], art['lr']
nombres_rf = rf_pipe.named_steps['prep'].get_feature_names_out()
imp_rf = pd.Series(rf_pipe.named_steps['clf'].feature_importances_, index=nombres_rf).sort_values(ascending=False)
print('Random Forest - importancia interna (por columna ya codificada):'); display(imp_rf.to_frame('importancia'))

nombres_lr = lr_pipe.named_steps['prep'].get_feature_names_out()
coef = pd.DataFrame(lr_pipe.named_steps['clf'].coef_, index=lr_pipe.named_steps['clf'].classes_, columns=nombres_lr)
print('Regresión logística - coeficientes por segmento (variables numéricas escaladas):'); display(coef.T)

### Limitaciones del modelo
1. **Solo 7 proveedores Ocasionales reales.** Las métricas de esa clase son muy ruidosas: un proveedor mal clasificado cambia el acierto en más de 14 puntos porcentuales por repetición. Con 3 particiones (el máximo posible) cada partición de prueba tiene 2 o 3 Ocasionales.
2. **Las variables disponibles casi no varían entre segmentos.** Calificación crediticia, proveedor preferido, activo, provincia y URL tienen muy poca variación o se reparten entre segmentos de forma parecida al total.
3. **Los 18 proveedores no tienen ninguna orden.** Asignarles un segmento es una extrapolación. Casi todos tienen `CreditRating = 1` y `PreferredVendorStatus = True` (se verifica en la celda que describe a los 18 proveedores).
4. **El segmento no es una clasificación formal.** Sale de un K-Means sobre dos variables (monto total y número de órdenes); cambia si cambian los datos o el número de grupos, y el modelo intenta predecir una etiqueta que depende de variables que no puede usar.
5. **Registros de prueba excluidos.** Los proveedores 1699 y 1700 se excluyeron solo del modelado (no de la segmentación de origen), por ser registros de prueba.
6. **Evaluación limitada.** 86 proveedores y 3 repeticiones de validación cruzada: las diferencias pequeñas entre modelos no son concluyentes. La importancia por permutación se midió sobre los datos de entrenamiento.

**Qué hace la celda siguiente:** resume los resultados principales con los números de esta ejecución, para pegarlos en las conclusiones del PR y de la HU.

In [ ]:
top = imp[mejor].iloc[0]
resumen_final = f'''Conclusiones HU 6.3
- Modelo elegido: {NOMBRE[mejor]} (precisión {R.loc[mejor,'accuracy_media']:.3f} ± {R.loc[mejor,'accuracy_std']:.3f}; F1 macro {R.loc[mejor,'f1_macro_media']:.3f} ± {R.loc[mejor,'f1_macro_std']:.3f}).
- Líneas base: mayoritaria {R.loc['mayoritaria','accuracy_media']:.3f} de precisión; azar estratificado {R.loc['estratificado','f1_macro_media']:.3f} de F1 macro.
- Supera ambas líneas base: {'sí' if supera_base else 'no'}; con claridad: {'sí' if supera_claro else 'no'}. Ganador claro entre modelos: {'no' if sin_ganador else 'sí'}.
- Acierto en Ocasionales (modelo elegido): {recall.loc['Ocasionales', NOMBRE[mejor]]:.2f} (7 proveedores reales).
- Variable con mayor importancia por permutación: {top['variable']} ({top['importancia_media']:.3f} ± {top['importancia_std']:.3f}).
- Los 18 proveedores: {int(out['baja_confianza'].sum())} de {len(out)} marcados con baja confianza (umbral {UMBRAL}).'''
print(resumen_final)